# EEG microstate clustering demo

This notebook is designed to run in JupyterLite in the browser. It uses a small synthetic EEG topography dataset so it does not depend on the local project data files.

The goal is to show the same idea as the project notebook: build topography samples, cluster them, and summarize the detected microstate groups.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(7)
n_channels = 64
n_samples = 300

def normalize_rows(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms


# Construct four canonical microstate topographies plus polarity flips.
base = np.array([
    np.linspace(0.9, -0.2, n_channels),
    np.sin(np.linspace(0, 3*np.pi, n_channels)),
    np.cos(np.linspace(0, 2*np.pi, n_channels)),
    np.linspace(-0.8, 0.8, n_channels),
])
microstates = np.vstack([base, -base])
labels = np.random.randint(0, len(microstates), size=n_samples)
X = microstates[labels] + 0.12 * np.random.randn(n_samples, n_channels)
X = normalize_rows(X)

print(f'Generated {n_samples} synthetic topographies across {len(microstates)} polarity-aware states.')
print('First row norm:', np.linalg.norm(X[0]))

model = HDBSCAN(min_cluster_size=20, cluster_selection_epsilon=0.05)
clusters = model.fit_predict(X)

print('Unique cluster labels:', sorted(set(clusters.tolist())))
print('Cluster assignment counts:')
print(pd.Series(clusters).value_counts().sort_index())

pca = PCA(n_components=2)
proj = pca.fit_transform(X)
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(proj[:, 0], proj[:, 1], c=clusters, cmap='viridis', s=25)
ax.set_title('PCA projection of synthetic microstate samples')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
fig.colorbar(scatter, ax=ax, label='cluster label')
fig.tight_layout()
plt.show()

# Show how polarity-invariant similarity treats a topography and its inversion as similar.
topo_a = X[0]
topo_b = -X[0]
sim = cosine_similarity(topo_a.reshape(1, -1), topo_b.reshape(1, -1))[0, 0]
print(f'Cosine similarity between +A and -A: {sim:.3f}')
print('This is close to -1 for exact polarity inversion, which is why polarity-invariant clustering is useful in microstate analysis.')

## What this demonstrates

- HDBSCAN can group repeated topographies into microstate-like clusters.
- PCA makes the cluster structure easier to inspect.
- A polarity-invariant comparison is the key concept behind EEG microstate analysis.

To run this online, open the notebook in JupyterLite with the direct link from the project README or JupyterLite landing page.